# Notebook 01 — Inspección inicial

**Objetivo:** Comprender la estructura y calidad inicial del dataset sin tomar decisiones definitivas. El propósito es reunir evidencia para orientar las etapas posteriores.

**Dataset:** `streaming_users_dirty.json` — registros de usuarios de una plataforma de streaming.

**Preguntas de análisis definidas por el grupo:**
1. ¿El plan de suscripción influye en el tiempo de visualización mensual?
2. ¿Los usuarios con mayor tiempo de visualización generan menos tickets de soporte?
3. ¿Existen perfiles diferenciados de usuarios según su comportamiento de consumo y soporte?

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

with open('../data/raw/streaming_users_dirty.json') as f:
    data = json.load(f)
df = pd.DataFrame(data)
print(f'Dataset cargado. Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

## 1. Estructura general

In [ ]:
print('Primeras filas:')
df.head(10)

In [ ]:
print('Tipos de datos y valores no nulos:')
df.info()

In [ ]:
print('Estadísticas descriptivas — variables numéricas:')
df.describe()

## 2. Variables identificadas

In [ ]:
for col in df.columns:
    print(f'{col}: tipo={df[col].dtype}, únicos={df[col].nunique()}, nulos={df[col].isnull().sum()}')

**Observación:** El dataset contiene 8 variables. Las numéricas son `age`, `monthly_watch_time_mins` y `customer_support_tickets`. Las categóricas son `subscription_plan`, `country` y `favorite_genre`. `last_login_date` es una variable de fecha almacenada como string. `user_id` es un identificador sin valor analítico.

## 3. Valores faltantes

In [ ]:
nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(2)
print(pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct})[nulos > 0])

**Observación:** Se detectaron nulos en `monthly_watch_time_mins` (193, 2.4%), `favorite_genre` (240, 2.9%) y `last_login_date` (320, 3.9%). Estas variables requerirán decisiones de imputación en la etapa de limpieza.

## 4. Duplicados

In [ ]:
n_dup = df.duplicated().sum()
print(f'Filas completamente duplicadas: {n_dup}')
print(df[df.duplicated()].head(3))

**Observación:** Se detectaron 126 filas completamente duplicadas. Deben eliminarse ya que representan registros repetidos que distorsionarían el análisis.

## 5. Inconsistencias en variables categóricas

In [ ]:
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f'\n{col} ({df[col].nunique()} valores únicos):')
    print(list(df[col].dropna().unique()))

**Observación:** Las tres variables categóricas presentan inconsistencias graves:
- `subscription_plan`: 15 variantes para 3 planes reales (Básico, Estándar, Premium). Ejemplos: 'Std', 'STANDARD', 'Basic', 'Premiun', 'estandar'.
- `country`: 26 variantes para 7 países. Ejemplos: 'Brazil'/'Brasil'/'brasil'/'BRA', 'Mexico'/'México'/'MEX'.
- `favorite_genre`: 29 variantes para 7 géneros. Ejemplos: 'CRIME'/'Crime'/'Crimen', 'comedy'/'Comedia'.

## 6. Outliers en variables numéricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['age', 'monthly_watch_time_mins', 'customer_support_tickets']):
    df[col].dropna().plot(kind='box', ax=ax, title=col)
plt.suptitle('Boxplots — detección inicial de outliers', y=1.02)
plt.tight_layout()
plt.show()

print('age: min =', df['age'].min(), '| max =', df['age'].max())
print('monthly_watch_time_mins: min =', df['monthly_watch_time_mins'].min(), '| max =', df['monthly_watch_time_mins'].max())
print('customer_support_tickets: min =', df['customer_support_tickets'].min(), '| max =', df['customer_support_tickets'].max())

**Observación:** Se detectaron valores claramente erróneos en las tres variables numéricas:
- `age`: valores -5 (imposible) y 130/150 (biológicamente inverosímiles).
- `monthly_watch_time_mins`: valores negativos (-120, -1) y extremos imposibles (50000, 99999).
- `customer_support_tickets`: valor -1 (imposible) y valores 99 y 150 (inverosímiles para soporte mensual).

## 7. Problema en last_login_date

In [ ]:
print('Muestra de valores en last_login_date:')
print(df['last_login_date'].dropna().sample(15, random_state=42).values)
slash = df['last_login_date'].dropna().str.contains('/').sum()
print(f'\nRegistros con formato YYYY/MM/DD: {slash}')

**Observación:** La columna `last_login_date` presenta dos formatos mezclados: `YYYY-MM-DD` y `YYYY/MM/DD`. Además se detectaron fechas futuras posteriores a la fecha actual, que son imposibles para un último login. Ambos problemas deben corregirse antes del análisis.

## 8. Resumen de observaciones iniciales

| Aspecto | Observación |
|---|---|
| Dimensiones | 8160 filas × 8 columnas |
| Duplicados | 126 filas completamente duplicadas |
| Nulos | monthly_watch_time_mins (193), favorite_genre (240), last_login_date (320) |
| Inconsistencias categóricas | subscription_plan (15 variantes), country (26 variantes), favorite_genre (29 variantes) |
| Outliers erróneos | age (-5, 130, 150), watch_time (negativos, 50000/99999), tickets (-1, 99, 150) |
| Fechas | Formatos mixtos y fechas futuras en last_login_date |

Estas observaciones orientan las decisiones de la etapa de calidad y limpieza.